In [ ]:
from pyspark.sql import functions as F

fact_sales = spark.table(
    "analytics_engineering.silver.fact_sales"
)

dim_product = spark.table(
    "analytics_engineering.silver.dim_product"
)

dim_subcategory = spark.table(
    "analytics_engineering.silver.dim_product_subcategory"
)

dim_category = spark.table(
    "analytics_engineering.silver.dim_product_category"
)

In [ ]:
sales_performance = (
    fact_sales.alias("f")
    .join(
        dim_product.alias("p"),
        F.col("f.ProductID") == F.col("p.ProductID"),
        "left"
    )
    .join(
        dim_subcategory.alias("s"),
        F.col("p.ProductSubcategoryID") ==
        F.col("s.ProductSubcategoryID"),
        "left"
    )
    .join(
        dim_category.alias("c"),
        F.col("s.ProductCategoryID") ==
        F.col("c.ProductCategoryID"),
        "left"
    )
    .select(
        F.col("f.SalesOrderID"),
        F.col("f.SalesOrderDetailID"),
        F.col("f.OrderDate"),
        F.col("f.CustomerID"),
        F.col("f.TerritoryID"),
        F.col("f.OnlineOrderFlag"),
        F.col("f.ProductID"),
        F.col("p.Name").alias("ProductName"),
        F.col("s.Name").alias("SubcategoryName"),
        F.col("c.Name").alias("CategoryName"),
        F.col("f.OrderQty").alias("UnitsSold"),
        F.col("f.UnitPrice"),
        F.col("f.UnitPriceDiscount"),
        F.col("f.LineTotal").alias("Revenue")
    )
)

display(sales_performance)

In [ ]:
gold_sales = (
    sales_performance
    .withColumn(
        "OrderDate",
        F.to_date("OrderDate")
    )
    .groupBy(
        "OrderDate",
        "CategoryName",
        "SubcategoryName",
        "ProductName",
        "OnlineOrderFlag"
    )
    .agg(
        F.sum("UnitsSold").alias("UnitsSold"),
        F.sum("Revenue").alias("Revenue"),
        F.countDistinct("SalesOrderID").alias("Orders"),
        F.countDistinct("CustomerID").alias("Customers"),
        F.sum("UnitPriceDiscount").alias("TotalDiscount")
    )
)

display(gold_sales)

In [ ]:
gold_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "analytics_engineering.gold.sales_performance"
    )

print("✓ gold.sales_performance created")

In [ ]:
display(
    spark.sql("""
        SHOW TABLES IN analytics_engineering.gold
    """)
)